In [22]:
from spin_lattices import KagomeLattice, SpinLattice, SquareLattice, TriangleLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from pathlib import Path
import torch
from torch_geometric.data import Data
from torch_geometric.data.batch import DataBatch
from torch_geometric.loader import DataLoader
from tqdm.auto import tqdm
from torch_geometric.nn import GCNConv, global_add_pool, GATConv
import torch.nn.functional as F
from torch_geometric.data import DataLoader
from torch.nn.functional import mse_loss
from torch.optim import Adam

In [2]:
lat = KagomeLattice(width=2, height=4)

In [3]:
system = HeisenbergJ1J2(
    lattice=lat,
    J1=1.0,
    J2=0.8,
    use_symmetries=True,
    spin_inversion=1,
    skip_symmetries_whitelist=True,
    ground_state_cache_dir=Path("groundstates")
)

2023-06-19 20:24:19.860 | DEBUG    | heisenberg_hamiltonians:__init__:446 - number_spins=24
2023-06-19 20:24:19.867 | DEBUG    | heisenberg_hamiltonians:__init__:456 - Symmetry group contains 16 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-06-19 20:24:19.956 | DEBUG    | heisenberg_hamiltonians:__init__:465 - Hilbert space dimension is 85662
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


In [4]:
system.get_eigenstates(1)

2023-06-19 20:24:20.040 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:60 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-0.8-True-1-1.pickle
2023-06-19 20:24:20.043 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:107 - Ground state energy is -40.5183420067


(array([-40.51834201]),
 array([[ 8.36163038e-08],
        [ 1.13414544e-07],
        [ 1.79331566e-08],
        ...,
        [-8.66655973e-03],
        [-1.17282633e-02],
        [ 3.99580256e-03]]))

In [5]:
df = (
    system.get_df_ground_state(unpack_configurations=True, expand_basis_columns=True)
    .drop("eigenstate_coeff", axis=1)
    .sample(n=50000)
    .assign(target=lambda df: df["amplitude"] / df["amplitude"].mean())
    .drop("amplitude", axis=1)
)

In [6]:
df

,s0,s1,s2,s3,s4,s5,s6,s7,s8,s9,...,s15,s16,s17,s18,s19,s20,s21,s22,s23,target
12134507,1,1,0,1,0,1,1,0,0,0,...,0,1,0,0,1,1,1,0,1,0.046551
13607840,0,0,0,0,0,1,0,1,1,1,...,1,1,1,1,1,0,0,1,1,0.062393
15016764,0,0,1,1,1,1,0,0,1,1,...,0,1,0,1,0,0,1,1,1,2.676538
15244389,1,0,1,0,0,1,1,0,0,0,...,1,0,0,0,1,0,1,1,1,5.827207
12644841,1,0,0,1,0,1,1,1,1,0,...,1,0,0,0,0,0,0,1,1,0.045144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15680547,1,1,0,0,0,1,0,0,0,0,...,0,1,1,1,1,0,1,1,1,0.047402
14235274,0,1,0,1,0,0,0,1,0,1,...,0,1,0,0,1,1,0,1,1,4.173705
2697662,0,1,1,1,1,1,0,1,1,0,...,0,1,0,0,1,0,1,0,0,0.004371
12847599,1,1,1,1,0,1,1,1,1,0,...,0,0,0,1,0,0,0,1,1,0.073326


In [7]:
# Get edge list
edge_list = torch.tensor(lat.as_igraph().get_edgelist(), dtype=torch.long).t().contiguous()

data_list = []

for i, row in tqdm(list(df.iterrows())):
    # Get node features
    node_features = torch.tensor(row.drop('target').values, dtype=torch.float).view(-1, 1)
    # Get node target
    node_target = torch.tensor([row['target']], dtype=torch.float)
    
    data = Data(x=node_features, edge_index=edge_list, y=node_target)
    data_list.append(data)


  0%|          | 0/50000 [00:00<?, ?it/s]

In [11]:
class Net(torch.nn.Module):
    def __init__(self, num_node_features):
        super(Net, self).__init__()
        self.conv1 = GCNConv(num_node_features, 128)
        self.conv2 = GCNConv(128, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = global_add_pool(x, batch)

        return torch.exp(x)

In [12]:
num_node_features = data_list[0].num_features  # Assuming all Data objects have the same number of features
model = Net(num_node_features)

In [14]:
def overlap(x, y):
    return torch.dot(x, y) / (torch.norm(x) * torch.norm(y))

In [37]:
# Create a DataLoader
loader = DataLoader(data_list, batch_size=32, shuffle=True)
full_data = DataBatch.from_data_list(data_list)

# Instantiate the model
num_node_features = data_list[0].num_features
model = Net(num_node_features)

# Use mean squared error loss for regression tasks
criterion = mse_loss

# Use Adam optimizer
optimizer = Adam(model.parameters(), lr=1e-1)

# Set device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Training loop
for epoch in range(100):
    total_loss = 0
    model.train()
    for data in loader:
        # Move data and target to the correct device
        data = data.to(device)
        target = data.y.to(device)

        # Forward pass
        out = model(data)
        loss = criterion(out, target.view(-1, 1))

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch: {epoch+1}, Loss: {total_loss/len(loader)}')
    
    print(f"{overlap(model(full_data).view(-1), full_data.y).item()=}")

Epoch: 1, Loss: 1.0380627786150126e+29
overlap(model(full_data).view(-1), full_data.y).item()=0.07593217492103577
Epoch: 2, Loss: 1.0383969501541956e+29
overlap(model(full_data).view(-1), full_data.y).item()=0.07593217492103577


KeyboardInterrupt: 

In [35]:
overlap(model(full_data).view(-1), full_data.y)

tensor(nan, grad_fn=<DivBackward0>)

In [38]:
model(full_data).view(-1)

tensor([6.4458e+12, 4.6790e+10, 2.2437e+11,  ..., 2.9442e+12, 5.6643e+13,
        2.4752e+13], grad_fn=<ViewBackward0>)